In [8]:
# ==============================================================================
# SCRIPT 05
# GENERACIÓN DEL DATASET DE ENTRENAMIENTO
#
# Proyecto:
# Clasificación jerárquica de coberturas forestales
# Predio SAGAMI
#
# PARTE 1
# ==============================================================================

import os
import csv
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from pathlib import Path
from rasterio.mask import mask

warnings.filterwarnings("ignore")

# ==============================================================================
# RUTAS
# ==============================================================================

GT_FILE = Path("/content/GroundTruth_SAGAMI_Final_v2.gpkg")

S2_FILE = Path("/content/Sentinel2_Composite_2025.tif")

S1_FILE = Path("/content/Sentinel1_Composite_2025.tif")

OUTPUT_FOLDER = Path("/content/Dataset")

OUTPUT_FOLDER.mkdir(exist_ok=True)

CSV_FILE = OUTPUT_FOLDER / "Dataset_SAGAMI.csv"

PARQUET_FILE = OUTPUT_FOLDER / "Dataset_SAGAMI.parquet"

# ==============================================================================
# CARGAR GROUND TRUTH
# ==============================================================================

print("="*80)
print("CARGANDO GROUND TRUTH")
print("="*80)

gt = gpd.read_file(GT_FILE)

print(f"Polígonos : {len(gt)}")
print(f"CRS       : {gt.crs}")

# ==============================================================================
# ABRIR SENTINEL-2
# ==============================================================================

print("\n" + "="*80)
print("CARGANDO SENTINEL-2")
print("="*80)

s2 = rasterio.open(S2_FILE)

print(f"Bandas : {s2.count}")
print(f"CRS    : {s2.crs}")
print(f"Pixel  : {s2.res}")

# ==============================================================================
# ABRIR SENTINEL-1
# ==============================================================================

print("\n" + "="*80)
print("CARGANDO SENTINEL-1")
print("="*80)

s1 = rasterio.open(S1_FILE)

print(f"Bandas : {s1.count}")
print(f"CRS    : {s1.crs}")
print(f"Pixel  : {s1.res}")

# ==============================================================================
# VALIDACIONES
# ==============================================================================

print("\n" + "="*80)
print("VALIDACIONES")
print("="*80)

assert gt.crs == s2.crs
assert gt.crs == s1.crs

assert s2.transform == s1.transform

assert s2.width == s1.width
assert s2.height == s1.height

assert s2.res == s1.res

print("✓ CRS")

print("✓ Transform")

print("✓ Resolución")

print("✓ Dimensiones")

# ==============================================================================
# CABECERA DEL DATASET
# ==============================================================================

columns = [

    "x",
    "y",

    "B2",
    "B3",
    "B4",
    "B5",
    "B6",
    "B7",
    "B8",
    "B8A",
    "B11",
    "B12",

    "VV",
    "VH",

    "id",

    "clase_n1",
    "clase_n2",
    "clase_n3"

]

# ==============================================================================
# CREAR CSV
# ==============================================================================

csv_file = open(
    CSV_FILE,
    mode="w",
    newline="",
    encoding="utf-8"
)

writer = csv.writer(csv_file)

writer.writerow(columns)

print("\nCSV creado correctamente.")

# ==============================================================================
# CONTADORES
# ==============================================================================

total_pixeles = 0

pixeles_validos = 0

pixeles_nodata = 0

print("\n" + "="*80)
print("LISTO PARA EXTRAER MUESTRAS")
print("="*80)

CARGANDO GROUND TRUTH
Polígonos : 61
CRS       : EPSG:32619

CARGANDO SENTINEL-2
Bandas : 10
CRS    : EPSG:32619
Pixel  : (10.0, 10.0)

CARGANDO SENTINEL-1
Bandas : 2
CRS    : EPSG:32619
Pixel  : (10.0, 10.0)

VALIDACIONES
✓ CRS
✓ Transform
✓ Resolución
✓ Dimensiones

CSV creado correctamente.

LISTO PARA EXTRAER MUESTRAS


In [11]:
# ==============================================================================
# SCRIPT 05A
# RASTERIZACIÓN DEL GROUND TRUTH
# ==============================================================================

import geopandas as gpd
import rasterio

from rasterio.features import rasterize

print("="*80)
print("SCRIPT 05A")
print("RASTERIZANDO GROUND TRUTH")
print("="*80)

GT_FILE="/content/GroundTruth_SAGAMI_Final_v2.gpkg"

REFERENCE="/content/Sentinel2_Composite_2025.tif"

OUTPUT="/content/GroundTruth_ID.tif"

# ==============================================================================
# CARGAR DATOS
# ==============================================================================

gt=gpd.read_file(GT_FILE)

ref=rasterio.open(REFERENCE)

print(f"Polígonos : {len(gt)}")

# ==============================================================================
# CREAR ID ÚNICO
# ==============================================================================

gt=gt.reset_index(drop=True)

gt["gt_id"]=gt.index+1

# ==============================================================================
# PREPARAR GEOMETRÍAS
# ==============================================================================

shapes=[

    (geom,idx)

    for geom,idx in zip(

        gt.geometry,

        gt.gt_id

    )

]

# ==============================================================================
# RASTERIZAR
# ==============================================================================

label=rasterize(

    shapes,

    out_shape=(

        ref.height,

        ref.width

    ),

    fill=0,

    transform=ref.transform,

    dtype="uint16"

)

# ==============================================================================
# GUARDAR
# ==============================================================================

profile=ref.profile.copy()

profile.update(

    dtype="uint16",

    count=1,

    compress="LZW"

)

with rasterio.open(

    OUTPUT,

    "w",

    **profile

) as dst:

    dst.write(

        label,

        1

    )

print()

print("="*80)

print("RESUMEN")

print("="*80)

print("Dimensiones")

print(label.shape)

print()

print("Pixeles etiquetados")

print((label>0).sum())

print()

print("Máximo ID")

print(label.max())

print()

print("Archivo generado")

print(OUTPUT)

# ==============================================================================
# GUARDAR TABLA DE IDs
# ==============================================================================

gt[["gt_id","clase_n1","clase_n2","clase_n3"]].to_csv(
    "/content/GroundTruth_ID_Table_.csv",
    index=False
)

print()

print("Tabla de IDs")

print("/content/GroundTruth_ID_Table.csv")

SCRIPT 05A
RASTERIZANDO GROUND TRUTH
Polígonos : 61

RESUMEN
Dimensiones
(394, 628)

Pixeles etiquetados
70246

Máximo ID
61

Archivo generado
/content/GroundTruth_ID.tif

Tabla de IDs
/content/GroundTruth_ID_Table.csv


In [12]:
# ==============================================================================
# SCRIPT 05B_v2
# PARTE 1
# GENERACIÓN DEL DATASET PIXEL A PIXEL
# ==============================================================================

import csv
import numpy as np
import pandas as pd
import rasterio

from pathlib import Path
from rasterio.transform import xy

# ==============================================================================
# RUTAS
# ==============================================================================

GT_RASTER = Path("/content/GroundTruth_ID.tif")

GT_TABLE = Path("/content/GroundTruth_ID_Table.csv")

S2_FILE = Path("/content/Sentinel2_Composite_2025.tif")

S1_FILE = Path("/content/Sentinel1_Composite_2025.tif")

OUTPUT_FOLDER = Path("/content/Dataset")

OUTPUT_FOLDER.mkdir(exist_ok=True)

CSV_FILE = OUTPUT_FOLDER / "Dataset_SAGAMI.csv"

# ==============================================================================
# CARGAR TABLA DE IDS
# ==============================================================================

print("="*80)
print("SCRIPT 05B_v2")
print("="*80)

print("\nCargando tabla de clases...")

gt_table = pd.read_csv(GT_TABLE)

print(gt_table.head())

# ==============================================================================
# DICCIONARIO
# ==============================================================================

print("\nCreando diccionario...")

lookup = {}

for _, row in gt_table.iterrows():

    lookup[int(row.gt_id)] = {

        "clase_n1": row.clase_n1,

        "clase_n2": row.clase_n2,

        "clase_n3": row.clase_n3

    }

print(f"{len(lookup)} clases cargadas.")

# ==============================================================================
# ABRIR RASTERS
# ==============================================================================

print("\nAbriendo raster...")

gt_src = rasterio.open(GT_RASTER)

s2_src = rasterio.open(S2_FILE)

s1_src = rasterio.open(S1_FILE)

gt = gt_src.read(1)

transform = gt_src.transform

print("Ground Truth :", gt.shape)

print("Sentinel-2   :", s2_src.count, "bandas")

print("Sentinel-1   :", s1_src.count, "bandas")

# ==============================================================================
# VALIDACIONES
# ==============================================================================

assert gt_src.transform == s2_src.transform
assert gt_src.transform == s1_src.transform

assert gt_src.crs == s2_src.crs
assert gt_src.crs == s1_src.crs

assert gt_src.width == s2_src.width == s1_src.width
assert gt_src.height == s2_src.height == s1_src.height

print("\nValidaciones OK.")

# ==============================================================================
# CREAR CSV
# ==============================================================================

csv_file = open(

    CSV_FILE,

    mode="w",

    newline="",

    encoding="utf-8"

)

writer = csv.writer(csv_file)

writer.writerow([

    "x",
    "y",

    "B2",
    "B3",
    "B4",
    "B5",
    "B6",
    "B7",
    "B8",
    "B8A",
    "B11",
    "B12",

    "VV",
    "VH",

    "gt_id",

    "clase_n1",
    "clase_n2",
    "clase_n3"

])

# ==============================================================================
# CONTADORES
# ==============================================================================

total_pixels = gt.size

valid_pixels = 0

nodata_pixels = 0

print()

print("="*80)

print("LISTO PARA EXTRAER MUESTRAS")

print("="*80)

SCRIPT 05B_v2

Cargando tabla de clases...
   gt_id           clase_n1                      clase_n2  \
0      1                NaN                     EUCALIPTO   
1      2                NaN                     EUCALIPTO   
2      3                NaN  20260105_NVJ_SIMAROUBA2_2025   
3      4                NaN   20260105_NVJ_SIMAROUBA_2025   
4      5  Ciclos Anteriores                            PC   

                                            clase_n3  
0  EUCALIPTO 2017 - NAVAJAS**SIN ANTECEDENTES DE ...  
1  EUCALIPTO 2017 - NAVAJAS* CON ANTECEDENTE DE S...  
2                       20260105_NVJ_SIMAROUBA2_2025  
3                        20260105_NVJ_SIMAROUBA_2025  
4                             PC_2004_talado_12.5has  

Creando diccionario...
61 clases cargadas.

Abriendo raster...
Ground Truth : (394, 628)
Sentinel-2   : 10 bandas
Sentinel-1   : 2 bandas

Validaciones OK.

LISTO PARA EXTRAER MUESTRAS


In [13]:
gt = gpd.read_file("/content/GroundTruth_SAGAMI_Final_v2.gpkg")

gt[gt["clase_n1"].isna()][
    ["clase_n1","clase_n2","clase_n3"]
]

,clase_n1,clase_n2,clase_n3
0,None,EUCALIPTO,EUCALIPTO 2017 - NAVAJAS**SIN ANTECEDENTES DE ...
1,None,EUCALIPTO,EUCALIPTO 2017 - NAVAJAS* CON ANTECEDENTE DE S...
2,None,20260105_NVJ_SIMAROUBA2_2025,20260105_NVJ_SIMAROUBA2_2025
3,None,20260105_NVJ_SIMAROUBA_2025,20260105_NVJ_SIMAROUBA_2025
5,None,PINO,PINO 2016 - NAVAJAS*
6,None,MIXTA,NAVAJAS MIXTA 2017
7,None,MIXTA,MIXTA NATIVAS 2023
8,None,PINO,NAVAJAS PINO 2017
16,None,SIMAROUBA,SIMAROUBA 2023
34,None,MIXTA,MIXTA 2016


In [14]:
import geopandas as gpd

GT = "/content/GroundTruth_SAGAMI_Final_v2.gpkg"

gt = gpd.read_file(GT)

# ------------------------------------------------------------------
# Completar clase_n1 cuando está vacía
# ------------------------------------------------------------------

mask = gt["clase_n1"].isna()

gt.loc[
    mask &
    gt["clase_n2"].isin([
        "PC",
        "EP",
        "EUCALIPTO",
        "PINO",
        "MIXTA",
        "SIMAROUBA",
        "20260105_NVJ_SIMAROUBA_2025",
        "20260105_NVJ_SIMAROUBA2_2025"
    ]),
    "clase_n1"
] = "Sembrado"

# ------------------------------------------------------------------
# Guardar nuevamente
# ------------------------------------------------------------------

gt.to_file(
    GT,
    driver="GPKG"
)

print("Valores faltantes en clase_n1:")
print(gt["clase_n1"].isna().sum())

print("\nConteo final:")
print(gt["clase_n1"].value_counts())

Valores faltantes en clase_n1:
0

Conteo final:
clase_n1
Sembrado             19
Buffer Bosque        12
Bosque                8
Areas netas           6
Pastos                5
Río                   3
Cultivo Arroz         3
Infrestructura        2
Ciclos Anteriores     1
Buffer laguna         1
Laguna                1
Name: count, dtype: int64


In [16]:
gt = gpd.read_file("/content/GroundTruth_SAGAMI_Final_v2.gpkg")

gt = gt.reset_index(drop=True)
gt["gt_id"] = gt.index + 1

gt[["gt_id","clase_n1","clase_n2","clase_n3"]].to_csv(
    "/content/GroundTruth_ID_Table__.csv",
    index=False
)

In [17]:
gt[gt["clase_n1"].isna()]

,id,predio,clase_n1,clase_n2,clase_n3,nombre_original,area_original_m2,area_buffer_m2,area_perdida_m2,porcentaje_perdido,clase_n3_original,geometry,gt_id


In [20]:
gt = gpd.read_file("/content/GroundTruth_SAGAMI_Final_v2.gpkg")

In [21]:
import rasterio

GT_RASTER = "/content/GroundTruth_ID.tif"

gt_src = rasterio.open(GT_RASTER)

gt_raster = gt_src.read(1)

transform = gt_src.transform

print(type(gt_raster))
print(gt_raster.shape)
print(gt_raster.dtype)

<class 'numpy.ndarray'>
(394, 628)
uint16


In [22]:
# ==============================================================================
# SCRIPT 05B_v2
# PARTE 1
# CONSTRUCCIÓN DEL DATASET
# ==============================================================================

import csv
import numpy as np
import pandas as pd
import rasterio

from pathlib import Path
from rasterio.transform import xy

print("="*80)
print("SCRIPT 05B_v2")
print("GENERACIÓN DEL DATASET")
print("="*80)

# ==============================================================================
# RUTAS
# ==============================================================================

GT_RASTER = Path("/content/GroundTruth_ID.tif")

GT_TABLE = Path("/content/GroundTruth_ID_Table.csv")

S2_FILE = Path("/content/Sentinel2_Composite_2025.tif")

S1_FILE = Path("/content/Sentinel1_Composite_2025.tif")

OUTPUT_FOLDER = Path("/content/Dataset")

OUTPUT_FOLDER.mkdir(exist_ok=True)

CSV_FILE = OUTPUT_FOLDER / "Dataset_SAGAMI.csv"

PARQUET_FILE = OUTPUT_FOLDER / "Dataset_SAGAMI.parquet"

# ==============================================================================
# TABLA DE CLASES
# ==============================================================================

print("\nCargando tabla de clases...")

gt_table = pd.read_csv(GT_TABLE)

print(gt_table.head())

# ==============================================================================
# DICCIONARIO
# ==============================================================================

lookup = {}

for _, r in gt_table.iterrows():

    lookup[int(r["gt_id"])] = {

        "clase_n1": r["clase_n1"],

        "clase_n2": r["clase_n2"],

        "clase_n3": r["clase_n3"]

    }

print(f"\nNúmero de clases: {len(lookup)}")

# ==============================================================================
# ABRIR RASTERS
# ==============================================================================

print("\nAbriendo Ground Truth...")

gt_src = rasterio.open(GT_RASTER)

gt_raster = gt_src.read(1)

transform = gt_src.transform

print("Abriendo Sentinel-2...")

s2_src = rasterio.open(S2_FILE)

print("Abriendo Sentinel-1...")

s1_src = rasterio.open(S1_FILE)

# ==============================================================================
# VALIDACIONES
# ==============================================================================

assert gt_src.crs == s2_src.crs == s1_src.crs

assert gt_src.transform == s2_src.transform == s1_src.transform

assert gt_src.width == s2_src.width == s1_src.width

assert gt_src.height == s2_src.height == s1_src.height

print("\nValidaciones OK")

# ==============================================================================
# LEER RASTERS
# ==============================================================================

print("\nLeyendo Sentinel-2...")

s2 = s2_src.read()

print("Leyendo Sentinel-1...")

s1 = s1_src.read()

print("\nSentinel-2:", s2.shape)

print("Sentinel-1:", s1.shape)

# ==============================================================================
# ASIGNAR BANDAS
# ==============================================================================

B2  = s2[0]
B3  = s2[1]
B4  = s2[2]
B5  = s2[3]
B6  = s2[4]
B7  = s2[5]
B8  = s2[6]
B8A = s2[7]
B11 = s2[8]
B12 = s2[9]

VV = s1[0]
VH = s1[1]

# ==============================================================================
# CREAR CSV
# ==============================================================================

csv_file = open(

    CSV_FILE,

    mode="w",

    newline="",

    encoding="utf-8"

)

writer = csv.writer(csv_file)

writer.writerow([

    "x",
    "y",

    "B2",
    "B3",
    "B4",
    "B5",
    "B6",
    "B7",
    "B8",
    "B8A",
    "B11",
    "B12",

    "VV",
    "VH",

    "gt_id",

    "clase_n1",
    "clase_n2",
    "clase_n3"

])

# ==============================================================================
# CONTADORES
# ==============================================================================

rows, cols = np.where(gt_raster > 0)

total_pixels = len(rows)

valid_pixels = 0

nodata_pixels = 0

print("\n"+"="*80)

print("RESUMEN")

print("="*80)

print("Pixeles etiquetados :", total_pixels)

print("\nLISTO PARA EXTRAER MUESTRAS")

SCRIPT 05B_v2
GENERACIÓN DEL DATASET

Cargando tabla de clases...
   gt_id           clase_n1                      clase_n2  \
0      1           Sembrado                     EUCALIPTO   
1      2           Sembrado                     EUCALIPTO   
2      3           Sembrado  20260105_NVJ_SIMAROUBA2_2025   
3      4           Sembrado   20260105_NVJ_SIMAROUBA_2025   
4      5  Ciclos Anteriores                            PC   

                                            clase_n3  
0  EUCALIPTO 2017 - NAVAJAS**SIN ANTECEDENTES DE ...  
1  EUCALIPTO 2017 - NAVAJAS* CON ANTECEDENTE DE S...  
2                       20260105_NVJ_SIMAROUBA2_2025  
3                        20260105_NVJ_SIMAROUBA_2025  
4                             PC_2004_talado_12.5has  

Número de clases: 61

Abriendo Ground Truth...
Abriendo Sentinel-2...
Abriendo Sentinel-1...

Validaciones OK

Leyendo Sentinel-2...
Leyendo Sentinel-1...

Sentinel-2: (10, 394, 628)
Sentinel-1: (2, 394, 628)

RESUMEN
Pixeles etiquetado

In [23]:
# ==============================================================================
# PARTE 2
# EXTRACCIÓN DE MUESTRAS
# ==============================================================================

print("\nIniciando extracción de muestras...\n")

for i, (row, col) in enumerate(zip(rows, cols), start=1):

    # ------------------------------------------------------------
    # Barra de progreso
    # ------------------------------------------------------------

    if i % 5000 == 0 or i == total_pixels:

        print(f"{i:,} / {total_pixels:,}")

    # ------------------------------------------------------------
    # ID del Ground Truth
    # ------------------------------------------------------------

    gt_id = int(gt_raster[row, col])

    info = lookup.get(gt_id)

    if info is None:
        continue

    # ------------------------------------------------------------
    # Coordenadas del centro del píxel
    # ------------------------------------------------------------

    x, y = xy(
        transform,
        row,
        col,
        offset="center"
    )

    # ------------------------------------------------------------
    # Sentinel-2
    # ------------------------------------------------------------

    b2  = float(B2[row, col])
    b3  = float(B3[row, col])
    b4  = float(B4[row, col])
    b5  = float(B5[row, col])
    b6  = float(B6[row, col])
    b7  = float(B7[row, col])
    b8  = float(B8[row, col])
    b8a = float(B8A[row, col])
    b11 = float(B11[row, col])
    b12 = float(B12[row, col])

    # ------------------------------------------------------------
    # Sentinel-1
    # ------------------------------------------------------------

    vv = float(VV[row, col])
    vh = float(VH[row, col])

    # ------------------------------------------------------------
    # Validar NoData
    # ------------------------------------------------------------

    valores = [

        b2,b3,b4,b5,b6,
        b7,b8,b8a,b11,b12,
        vv,vh

    ]

    if np.isnan(valores).any():

        nodata_pixels += 1
        continue

    # ------------------------------------------------------------
    # Escribir directamente al CSV
    # ------------------------------------------------------------

    writer.writerow([

        x,
        y,

        b2,
        b3,
        b4,
        b5,
        b6,
        b7,
        b8,
        b8a,
        b11,
        b12,

        vv,
        vh,

        gt_id,

        info["clase_n1"],
        info["clase_n2"],
        info["clase_n3"]

    ])

    valid_pixels += 1

print("\nExtracción finalizada.")


Iniciando extracción de muestras...

5,000 / 70,246
10,000 / 70,246
15,000 / 70,246
20,000 / 70,246
25,000 / 70,246
30,000 / 70,246
35,000 / 70,246
40,000 / 70,246
45,000 / 70,246
50,000 / 70,246
55,000 / 70,246
60,000 / 70,246
65,000 / 70,246
70,000 / 70,246
70,246 / 70,246

Extracción finalizada.


In [24]:
# ==============================================================================
# PARTE 3
# CERRAR CSV Y VALIDAR DATASET
# ==============================================================================

print("\nCerrando archivo CSV...")

csv_file.close()

print("Archivo CSV cerrado correctamente.")

# ==============================================================================
# CARGAR DATASET
# ==============================================================================

print("\nLeyendo Dataset generado...")

dataset = pd.read_csv(CSV_FILE)

# ==============================================================================
# EXPORTAR PARQUET
# ==============================================================================

dataset.to_parquet(
    PARQUET_FILE,
    index=False
)

# ==============================================================================
# RESUMEN
# ==============================================================================

print("\n" + "="*80)
print("RESUMEN FINAL")
print("="*80)

print(f"Muestras válidas : {len(dataset):,}")

print(f"NoData descartados : {nodata_pixels:,}")

print(f"CSV : {CSV_FILE}")

print(f"Parquet : {PARQUET_FILE}")

print("\nPrimeras filas")

display(dataset.head())

print("\nInformación")

print(dataset.info())

print("\nValores nulos")

print(dataset.isna().sum())

print("\nDistribución Nivel 1")

print(dataset["clase_n1"].value_counts())

print("\nDistribución Nivel 2")

print(dataset["clase_n2"].value_counts())

print("\nDistribución Nivel 3")

print(dataset["clase_n3"].value_counts())

print("\nEstadísticas numéricas")

display(dataset.describe())


Cerrando archivo CSV...
Archivo CSV cerrado correctamente.

Leyendo Dataset generado...

RESUMEN FINAL
Muestras válidas : 70,230
NoData descartados : 16
CSV : /content/Dataset/Dataset_SAGAMI.csv
Parquet : /content/Dataset/Dataset_SAGAMI.parquet

Primeras filas


,x,y,B2,B3,B4,B5,B6,B7,B8,B8A,B11,B12,VV,VH,gt_id,clase_n1,clase_n2,clase_n3
0,61995.0,446585.0,927.0,1123.0,1248.0,1344.0,1185.5,1216.0,1064.0,1061.5,243.0,185.0,-22.534081,-28.822645,59,Río,R,Río
1,62005.0,446585.0,951.0,1140.0,1278.0,1402.0,1182.0,1256.0,1078.0,1118.0,228.0,185.0,-22.424263,-29.092960,59,Río,R,Río
2,62015.0,446585.0,960.0,1136.0,1274.0,1402.0,1182.0,1256.0,1078.0,1118.0,228.0,185.0,-22.537445,-27.733332,59,Río,R,Río
3,62025.0,446585.0,954.0,1142.0,1266.0,1425.0,1234.0,1240.0,1084.0,1149.0,244.0,194.0,-22.816227,-28.505375,59,Río,R,Río
4,62035.0,446585.0,954.0,1180.0,1288.0,1425.0,1234.0,1240.0,1094.0,1149.0,244.0,194.0,-22.205042,-28.210127,59,Río,R,Río



Información
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70230 entries, 0 to 70229
Data columns (total 18 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   x         70230 non-null  float64
 1   y         70230 non-null  float64
 2   B2        70230 non-null  float64
 3   B3        70230 non-null  float64
 4   B4        70230 non-null  float64
 5   B5        70230 non-null  float64
 6   B6        70230 non-null  float64
 7   B7        70230 non-null  float64
 8   B8        70230 non-null  float64
 9   B8A       70230 non-null  float64
 10  B11       70230 non-null  float64
 11  B12       70230 non-null  float64
 12  VV        70230 non-null  float64
 13  VH        70230 non-null  float64
 14  gt_id     70230 non-null  int64  
 15  clase_n1  70230 non-null  object 
 16  clase_n2  70230 non-null  object 
 17  clase_n3  70230 non-null  object 
dtypes: float64(14), int64(1), object(3)
memory usage: 9.6+ MB
None

Valores nulos
x           0
y

,x,y,B2,B3,B4,B5,B6,B7,B8,B8A,B11,B12,VV,VH,gt_id
count,70230.000000,70230.000000,70230.000000,70230.000000,70230.000000,70230.000000,70230.000000,70230.000000,70230.000000,70230.000000,70230.000000,70230.000000,70230.000000,70230.000000,70230.000000
mean,63674.284067,444398.448384,588.204727,838.680158,777.809789,1392.335519,2526.740994,2958.349267,2935.247280,3275.310117,2605.915307,1469.540859,-10.111042,-16.040347,39.098733
std,1815.996042,773.074352,145.780306,201.937757,330.539606,363.641790,536.526874,639.725129,666.666279,726.020111,880.765208,623.571705,2.340933,2.524735,14.264781
min,60835.000000,442675.000000,183.500000,291.500000,290.500000,331.000000,74.000000,77.500000,43.000000,28.000000,100.000000,72.000000,-23.949461,-30.189369,1.000000
25%,62215.000000,443855.000000,447.000000,656.000000,432.000000,1092.500000,2411.500000,2822.000000,2792.000000,3150.500000,1941.000000,891.500000,-10.928497,-16.677277,29.000000
50%,62965.000000,444335.000000,593.000000,850.500000,797.000000,1433.000000,2616.000000,3068.000000,3050.000000,3416.500000,2634.000000,1460.000000,-9.821735,-15.539048,45.000000
75%,65655.000000,444845.000000,710.000000,1004.000000,1056.000000,1671.500000,2796.000000,3293.000000,3284.000000,3646.500000,3348.000000,2011.000000,-8.576422,-14.548680,50.000000
max,67095.000000,446585.000000,1132.000000,1638.000000,2050.000000,2670.000000,3826.000000,4547.000000,4940.000000,4946.500000,4784.000000,3246.000000,-5.849859,-11.894634,61.000000
